# Phase 1: NEU-DET defect detector: train, evaluate, export TFLite (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajaybahuguna17/qc-defect-detector/blob/master/phase1-vision/notebooks/train_colab.ipynb)

Runs end to end on a **fresh Colab GPU runtime**:

1. clone repo, install pinned deps (`phase1-vision/requirements.txt`)
2. download NEU-DET, `prepare_data.py` (seeded 80/20 split, zero train/val overlap enforced)
3. train YOLOv8n at `imgsz=320` (NEU images are 200x200), fixed seed
4. validate: mAP50, mAP50-95, per-class P/R, confusion matrix PNG
5. export TFLite **FP32** and **INT8** (INT8 calibrated on train images), assert 6-class output `[1, 10, N]`
6. run `.pt`, FP32 `.tflite` and INT8 `.tflite` on the same 10 val images and compare detections
7. mAP of each TFLite on val
8. deterministic severity rule demo (`phase1-vision/severity.py`)
9. save `best.pt`, both `.tflite`, `labels.txt`, `model_io.json`, results JSON and plots to Google Drive

**How to run:** `Runtime -> Change runtime type -> GPU (T4 is fine)`, check the config cell, then `Runtime -> Run all`.
Nothing falls back silently: any missing file or failed step raises an error.

Note on `yolov8n.pt`: training starts from the official COCO-pretrained YOLOv8n weights (transfer learning).
That is the only place they are used. Everything evaluated and exported is the NEU-DET `best.pt` trained here.

## 0. Configuration

In [ ]:
# ---- edit here --------------------------------------------------------------
REPO_URL   = "https://github.com/ajaybahuguna17/qc-defect-detector.git"
BRANCH     = "master"
EPOCHS     = 100          # default 100
PATIENCE   = 30           # early-stop patience (epochs without val improvement)
IMGSZ      = 320          # NEU-DET images are 200x200
BATCH      = 32
SEED       = 42           # dataset split, training, sample selection
N_COMPARE  = 10           # val images used for the .pt vs .tflite comparison
USE_DRIVE  = True         # copy all outputs to Google Drive
DRIVE_DIR  = "/content/drive/MyDrive/qc-defect-detector/phase1"
ALLOW_CPU  = False        # refuse to train without a GPU unless you set this True
# ------------------------------------------------------------------------------
import os, sys
from pathlib import Path
REPO_DIR    = Path("/content/qc-defect-detector")
PHASE1      = REPO_DIR / "phase1-vision"
WORK        = Path("/content/work")                 # runs/ and downloaded yolov8n.pt live here, not in the repo
RUN_NAME    = f"neu_yolov8n_{IMGSZ}_e{EPOCHS}_s{SEED}"
EXPORT_DIR  = WORK / "export"
RESULTS_DIR = WORK / "results"
for d in (WORK, EXPORT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("run name:", RUN_NAME)

## 1. GPU check and Google Drive

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU, then Run all again (or set ALLOW_CPU=True).")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("device:", GPU_NAME)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    print("Drive output dir:", DRIVE_DIR)

## 2. Clone the repo and install pinned dependencies
If the repo is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left bar).

If pip prints *"You must restart the runtime"*, do `Runtime -> Restart session` and then `Run all` again (all cells are safe to re-run).

In [ ]:
import shutil
def run(cmd, **kw):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, **kw)   # raises on failure

url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass

if REPO_DIR.exists():
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"])
else:
    run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR])
GIT_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", GIT_COMMIT)

run(["pip", "install", "-q", "-r", PHASE1 / "requirements.txt"])

import importlib.metadata as md_
VERSIONS = {}
for pkg in ["ultralytics", "litert-torch", "ai-edge-litert", "ai-edge-quantizer", "torch", "torchvision",
            "numpy", "opencv-python", "opencv-python-headless", "pillow", "matplotlib"]:
    try:
        VERSIONS[pkg] = md_.version(pkg)
    except md_.PackageNotFoundError:
        pass
import platform
VERSIONS["python"] = sys.version.split()[0]
VERSIONS["cuda"] = torch.version.cuda
VERSIONS["platform"] = platform.platform()
for k, v in VERSIONS.items():
    print(f"{k:24s} {v}")
assert VERSIONS["ultralytics"] == "8.4.198", "unexpected ultralytics version; requirements.txt not applied"

## 3. Download NEU-DET and build the YOLO dataset (`prepare_data.py`)

In [ ]:
run([sys.executable, PHASE1 / "scripts" / "prepare_data.py",
     "--raw-dir", PHASE1 / "data" / "raw",
     "--dataset-dir", PHASE1 / "data" / "dataset",
     "--seed", SEED, "--split-ratio", 0.8])

DATA_YAML = (PHASE1 / "data" / "dataset" / "data.yaml").resolve()
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
def images(split):
    return sorted(p for p in (PHASE1 / "data" / "dataset" / "images" / split).iterdir() if p.suffix.lower() in IMG_EXT)
train_imgs, val_imgs = images("train"), images("val")
overlap = {p.stem for p in train_imgs} & {p.stem for p in val_imgs}
assert not overlap, f"train/val overlap: {sorted(overlap)[:10]}"
assert len(train_imgs) + len(val_imgs) == 1800, f"expected 1800 NEU-DET images, got {len(train_imgs) + len(val_imgs)}"
DATASET = {"train_images": len(train_imgs), "val_images": len(val_imgs), "overlap": 0, "split_seed": SEED}
print(DATASET)
print(DATA_YAML.read_text())

## 4. Train YOLOv8n (`imgsz=320`, fixed seed)

In [ ]:
import time
from ultralytics import YOLO

t0 = time.time()
model = YOLO("yolov8n.pt")          # COCO-pretrained starting point (transfer learning), downloaded into /content/work
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    seed=SEED, deterministic=True, workers=2,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True, plots=True,
)
TRAIN_MIN = (time.time() - t0) / 60
TRAIN_DIR = Path(model.trainer.save_dir)
BEST_PT = TRAIN_DIR / "weights" / "best.pt"
if not BEST_PT.is_file():
    raise FileNotFoundError(f"training finished but {BEST_PT} is missing")
print(f"trained in {TRAIN_MIN:.1f} min -> {BEST_PT}")

## 5. Validate `best.pt` on val: mAP, per-class P/R, confusion matrix

In [ ]:
from IPython.display import Image as IPImage, display

best = YOLO(str(BEST_PT))
CLASS_NAMES = [best.names[i] for i in sorted(best.names)]
assert len(CLASS_NAMES) == 6, CLASS_NAMES

VAL_DIR = WORK / "runs" / f"{RUN_NAME}_val_pt"
m = best.val(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, split="val", plots=True,
             project=str(WORK / "runs"), name=VAL_DIR.name, exist_ok=True)
VAL_DIR = Path(getattr(m, "save_dir", VAL_DIR))

per_class = {}
for i, c in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    per_class[CLASS_NAMES[int(c)]] = {"precision": float(p), "recall": float(r), "mAP50": float(ap50), "mAP50_95": float(ap)}
PT_METRICS = {"mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
              "precision": float(m.box.mp), "recall": float(m.box.mr), "per_class": per_class}

print(f"mAP50 {PT_METRICS['mAP50']:.4f} | mAP50-95 {PT_METRICS['mAP50_95']:.4f} | P {PT_METRICS['precision']:.4f} | R {PT_METRICS['recall']:.4f}")
print(f"{'class':18s} {'P':>7s} {'R':>7s} {'mAP50':>7s} {'mAP50-95':>9s}")
for k, v in per_class.items():
    print(f"{k:18s} {v['precision']:7.3f} {v['recall']:7.3f} {v['mAP50']:7.3f} {v['mAP50_95']:9.3f}")

CM_PNG = VAL_DIR / "confusion_matrix.png"
CM_NORM_PNG = VAL_DIR / "confusion_matrix_normalized.png"
if not CM_PNG.is_file():
    raise FileNotFoundError(f"confusion matrix not written: {CM_PNG}")
display(IPImage(str(CM_NORM_PNG if CM_NORM_PNG.is_file() else CM_PNG), width=600))

## 6. Export TFLite FP32 + INT8 (`export_tflite.py`)
INT8 is calibrated on ~30% of the **train** images so the val set stays unseen for evaluation.
The script raises if the export fails or if the output is not `[1, 10, N]`.

In [ ]:
run([sys.executable, PHASE1 / "scripts" / "export_tflite.py",
     "--weights", BEST_PT, "--data", DATA_YAML, "--imgsz", IMGSZ,
     "--out-dir", EXPORT_DIR, "--precisions", "fp32,int8", "--calib-split", "train", "--calib-fraction", 0.3])

import json
MODEL_IO = json.loads((EXPORT_DIR / "model_io.json").read_text())
TFLITE = {p: EXPORT_DIR / MODEL_IO["models"][p]["file"] for p in ("fp32", "int8")}
for p, f in TFLITE.items():
    io = MODEL_IO["models"][p]
    assert f.is_file(), f
    assert io["output_shape"][1] == 10, io["output_shape"]
    print(f"{p}: {f.name} {io['size_bytes']/1e6:.2f} MB | input {io['input_shape']} {io['input_layout']} {io['input_dtype']} "
          f"| output {io['output_shape']} {io['output_dtype']}")

## 7. Same 10 val images through `.pt`, FP32 `.tflite`, INT8 `.tflite`: compare detections

In [ ]:
import random, numpy as np
sys.path.insert(0, str(PHASE1 / "scripts"))
sys.path.insert(0, str(PHASE1))
from tflite_utils import load_interpreter, describe_io, run_tflite_on_image, match_detections

CONF, IOU = 0.25, 0.45     # same thresholds for all three models
sample = random.Random(SEED).sample(val_imgs, N_COMPARE)

def pt_detect(path):
    r = best.predict(str(path), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
    return [{"class_id": int(c), "class_name": CLASS_NAMES[int(c)], "confidence": float(s), "bbox_xyxy": [float(v) for v in b]}
            for b, s, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy())]

interps = {p: load_interpreter(f) for p, f in TFLITE.items()}
ios = {p: describe_io(it) for p, it in interps.items()}

COMPARE, box_row_max = [], {p: 0.0 for p in TFLITE}
print(f"{'image':22s} {'pt':>3s} | {'fp32 n/match/IoU/dconf':>26s} | {'int8 n/match/IoU/dconf':>26s}")
for img in sample:
    ref = pt_detect(img)
    row = {"image": img.name, "pt": ref}
    cells = []
    for p in TFLITE:
        dets, raw = run_tflite_on_image(interps[p], ios[p], img, CLASS_NAMES, CONF, IOU)
        box_row_max[p] = max(box_row_max[p], float(np.abs(raw[0, :4]).max()))
        mt = match_detections(ref, dets)
        row[p] = {"detections": dets, "match_vs_pt": mt}
        fmt = lambda x: "-" if x is None else f"{x:.3f}"
        cells.append(f"{mt['n_other']:>3d}/{mt['matched']:>2d}/{fmt(mt['mean_iou'])}/{fmt(mt['mean_abs_conf_diff'])}")
    COMPARE.append(row)
    print(f"{img.name:22s} {len(ref):>3d} | {cells[0]:>26s} | {cells[1]:>26s}")

def summary(p):
    n_ref = sum(len(r["pt"]) for r in COMPARE)
    matched = sum(r[p]["match_vs_pt"]["matched"] for r in COMPARE)
    n_other = sum(r[p]["match_vs_pt"]["n_other"] for r in COMPARE)
    return {"pt_detections": n_ref, "tflite_detections": n_other, "matched": matched,
            "match_rate": matched / n_ref if n_ref else None, "max_abs_box_output": box_row_max[p]}
COMPARE_SUMMARY = {p: summary(p) for p in TFLITE}
print(json.dumps(COMPARE_SUMMARY, indent=2))
for p, s in COMPARE_SUMMARY.items():
    print(f"{p}: raw box rows max |value| = {s['max_abs_box_output']:.3f} -> coordinates are "
          f"{'NORMALISED 0..1' if s['max_abs_box_output'] <= 1.5 else 'in INPUT PIXELS'} (record this in MODEL_IO.md)")
s = COMPARE_SUMMARY["fp32"]
if s["pt_detections"] >= 5 and s["match_rate"] < 0.9:
    raise AssertionError(f"FP32 TFLite disagrees with .pt (match rate {s['match_rate']:.2f}); decoder or export is wrong")

## 8. mAP of each TFLite on val (Ultralytics validator, LiteRT on CPU)

In [ ]:
TFLITE_METRICS = {}
for p, f in TFLITE.items():
    mt = YOLO(str(f), task="detect").val(data=str(DATA_YAML), imgsz=IMGSZ, batch=1, split="val", plots=False,
                                        project=str(WORK / "runs"), name=f"{RUN_NAME}_val_{p}", exist_ok=True, verbose=False)
    TFLITE_METRICS[p] = {"mAP50": float(mt.box.map50), "mAP50_95": float(mt.box.map),
                         "precision": float(mt.box.mp), "recall": float(mt.box.mr)}

print(f"{'model':8s} {'mAP50':>7s} {'mAP50-95':>9s} {'P':>7s} {'R':>7s}")
for name, v in [("pt", PT_METRICS), *TFLITE_METRICS.items()]:
    print(f"{name:8s} {v['mAP50']:7.4f} {v['mAP50_95']:9.4f} {v['precision']:7.4f} {v['recall']:7.4f}")
for p, v in TFLITE_METRICS.items():
    print(f"{p}: mAP50 drop vs .pt = {PT_METRICS['mAP50'] - v['mAP50']:+.4f}")

## 9. Severity rule demo (`phase1-vision/severity.py`)
Deterministic: class + bbox area ratio + confidence + defect count. Thresholds live in `SEVERITY_CONFIG`.
Runs on the FP32 TFLite detections, i.e. exactly what the phone will see.

In [ ]:
import importlib, severity
importlib.reload(severity)
from severity import assess_severity, SEVERITY_CONFIG
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

print(json.dumps(SEVERITY_CONFIG, indent=2))
SEVERITY_DEMO = []
fig, axes = plt.subplots(2, (N_COMPARE + 1) // 2, figsize=(4 * ((N_COMPARE + 1) // 2), 8))
for ax, row, img in zip(axes.ravel(), COMPARE, sample):
    w, h = Image.open(img).size
    dets = row["fp32"]["detections"]
    sev = assess_severity(dets, w, h)
    SEVERITY_DEMO.append({"image": img.name, **sev.to_dict()})
    ax.imshow(Image.open(img).convert("RGB"))
    for d in dets:
        x1, y1, x2, y2 = d["bbox_xyxy"]
        keep = d["confidence"] >= SEVERITY_CONFIG["min_confidence"]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1.5, color="red" if keep else "gray"))
        ax.text(x1, y1, f"{d['class_name']} {d['confidence']:.2f}", fontsize=6, color="yellow", va="bottom")
    ax.set_title(f"{img.name}\nseverity: {sev.level} (n={sev.defect_count})", fontsize=8)
    ax.axis("off")
    print(f"{img.name:22s} -> {str(sev.level):8s} | {sev.reason}")
plt.tight_layout()
SEVERITY_PNG = RESULTS_DIR / "severity_demo.png"
plt.savefig(SEVERITY_PNG, dpi=120)
plt.show()

## 10. Save everything (results JSON, plots, models) and copy to Google Drive

In [ ]:
from datetime import datetime, timezone

RESULTS = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT,
    "gpu": GPU_NAME,
    "versions": VERSIONS,
    "config": {"epochs": EPOCHS, "patience": PATIENCE, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
               "base_weights": "yolov8n.pt (COCO-pretrained, transfer learning)", "compare_conf": CONF, "compare_iou": IOU},
    "train_minutes": round(TRAIN_MIN, 1),
    "epochs_completed": int(getattr(model.trainer, "epoch", -1)) + 1,
    "dataset": DATASET,
    "classes": CLASS_NAMES,
    "val_pt": PT_METRICS,
    "val_tflite": TFLITE_METRICS,
    "model_io": MODEL_IO,
    "compare_10_images": {"summary": COMPARE_SUMMARY, "per_image": COMPARE},
    "severity_config": SEVERITY_CONFIG,
    "severity_demo": SEVERITY_DEMO,
}
(RESULTS_DIR / "phase1_results.json").write_text(json.dumps(RESULTS, indent=2))

# plots
for src in [CM_PNG, CM_NORM_PNG, *VAL_DIR.glob("*curve*.png"), TRAIN_DIR / "results.png", TRAIN_DIR / "results.csv"]:
    if Path(src).is_file():
        shutil.copy2(src, RESULTS_DIR / Path(src).name)

# models
MODELS_OUT = WORK / "models"
MODELS_OUT.mkdir(exist_ok=True)
shutil.copy2(BEST_PT, MODELS_OUT / f"best_{RUN_NAME}.pt")
for f in [*TFLITE.values(), EXPORT_DIR / "labels.txt", EXPORT_DIR / "model_io.json"]:
    shutil.copy2(f, MODELS_OUT / f.name)

if USE_DRIVE:
    dest = Path(DRIVE_DIR) / RUN_NAME
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(RESULTS_DIR, dest / "results")
    shutil.copytree(MODELS_OUT, dest / "models")
    print("Saved to Drive:", dest)
for p in sorted((WORK / "models").iterdir()) + sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.relative_to(WORK)}  ({p.stat().st_size/1e6:.2f} MB)")

## 11. After the run
1. From Drive `.../<run>/results/`, commit `phase1_results.json` and the PNGs to `phase1-vision/results/` (small files).
   Do **not** commit `.pt` / `.tflite` (gitignored); share them via Drive or a GitHub Release.
2. Fill the TODOs in `phase1-vision/MODEL_IO.md` from `models/model_io.json` and the coordinate line printed in step 7.
3. Update the Phase 1 tracker in `PROJECT_CONTEXT.md` with the real numbers.